# Support Vector Machines

# 1. Introduction

The task at hand is binary classification. Given labeled data points belonging to 2 classes, we need to find a rule that can distinguish them. 

$x \in R^{N \times D}$ - matrix with data points \
$y \in R^D$ - vector of labels \
$D$ - feature dimension \
$N$ - number of data points \

The way support vector machine, perceptron, and other linear models work is by creating a D-dimensional hyperplane that separates the points belonging to different classes. 

First, let's generate some data and draw lines (2D hyperplanes) that separate it.




In [ ]:
import numpy as np
from numpy.random import rand
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def get_data(all_points, separable=True):
  """
  :param all_pojnts: number of data points to be generated
  :param separable: boolean determining whether the generated data is linearly separable
  :return: a tuple (x, y):
  - x - numpy array of size (all_points, 2) with the data points
  - y - numpy array of size (all_points, ) with the data labels (1 or -1)
  """

  size = all_points // 2

  if separable:
    b1, b2 = 1.3, 0.7
  else:
    b1, b2 = 0.5, 0.3
  
  x0 = np.concatenate((rand(size), rand(size) + b1))
  x1 = np.concatenate((rand(size), rand(size) + b2))
  
  label = np.ones(all_points)
  label[:size] = -1
  return np.stack((x0, x1),axis=1), label

def get_random_line(magnitude=3):
  """
  Hesse normal form : x cos θ + y sin θ - p = 0
  where the line is orthogonal to a vector of length p and angle θ
  nice for generating random lines

  :param magnitude: scalar magnitude of largest random vector defining the line
  :return: a tuple (w, b):
  - w: a numpy array of size 2 with coefficients of the line
  - b: constant of the line
  """

  theta = rand() * 2 * np.pi
  w = np.array([np.cos(theta), np.sin(theta)])
  b = - rand() * magnitude
  return w, b

def lineary_separates_2D(x, y, w, b):
  """
  :return: boolean determining whether the line separates the data points correctly
  """
  pred = x.dot(w)+ b
  return np.all(( pred > 0 ) == ( y > 0))

def plot_data(x, y, margin=0.1):
  """
  Plots all the data points onto a graph
  
  :return: a tuple (x_min, x_max):
  - x_min: plot's minimum x coordinate value
  - x_max: plot's maximum x coordinate value
  """

  plot_dict = {'x0' : x[:,0], 'x1': x[:,1], 'label':y}
  sns.relplot(data=plot_dict, x='x0', y='x1', hue='label', 
            s=200, height=7, aspect=2, palette=sns.color_palette("dark",2) )
  x_min, x_max = min(plot_dict['x0']) - margin, max(plot_dict['x0']) + margin
  y_min, y_max = min(plot_dict['x1']) - margin, max(plot_dict['x1']) + margin
  plt.xlim(x_min, x_max)
  plt.ylim(y_min, y_max)
  return x_min, x_max

def draw_line(w, b, x_min, x_max, color='green'):
  """
  Plots a line onto a graph in form:  w[0] * x + w[1] * y + b = 0

  :param w: a numpy array of size 2 with coefficients of the line
  :param b: constant of the line
  :param x_min: float determining the minimum x value for drawing
  :param x_max: float determining the maximum x value for drawing
  :param color: string determining color of the line
  """

  y_l = - (x_min * w[0] / w[1] + b / w[1])
  y_r = - (x_max * w[0] / w[1] + b / w[1])
  plt.plot((x_min, x_max), (y_l, y_r), linewidth=1, color=color)


# Exercise 1
In the code below, you can experiment with different random seed, the number of data points and lines to observe the space of possible solutions and how it depends on the distribution of data points.

In [ ]:
np.random.seed(1031)

x, y = get_data(all_points=32, separable=True)

x_min, x_max = plot_data(x, y)

# draw {n_lines} lines that seperate the data
n_lines = 10
for _ in range(n_lines):
  w, b = get_random_line()
  while not lineary_separates_2D(x, y, w, b):
    w, b = get_random_line()
  draw_line(w, b , x_min, x_max)

# 2. Hyperplanes and margins

As seen in the example above if the data is linearly separable, there are usually a lot of different lines that partition it. 
 *Which one is the best?*


**Support Vector Machines** (SVM) provides an answer to this question. Like perceptron, they are binary classification models defined by the hyperplane that separates the data. 
The hyperplane is defined by the equation: $$xw-b=0$$
where $x\in R^{N \times D}, w \in R^D$, $b\in R$ and $D$ is dimension of the data.
Thus the main difference between perceptron and SVM is the way the weights (which define the hyperplane) are learned. 

First, two additional parallel hyperplanes known as margins are defined:
$$xw-b=1$$
$$xw-b=-1$$

when $x_iw-b \geq 1$ clasify the point $y_i=1$

when $x_iw-b \leq -1$ clasify the point $y_i=-1$

The objective is to find the hyperplane that separates the data with the largest distance between the margins. This version of SVM is called hard margin since the margins are not violated.

\

Run the next 2 cells to generate a hyperplane with margins.

In [ ]:
# you don't need to understand this code, it is just used to generate random lines with valid margins

def get_random_direction():
  """
  :return: a numpy array of size (2,) with a vector of size 1 with random direction
  """

  theta = rand() * 2 * np.pi
  return np.array([np.cos(theta), np.sin(theta)])

def direction_lineary_separates_2D(x, y, w):
  """
  :return: boolean determining whether there exists a line with direction determined 
  by w which separates the data points correctly
  """
  projected = x.dot(w)
  l = np.max( projected[ y == -1 ] )
  r = np.min( projected[ y ==  1 ] )
  return l < r

def find_line_with_margins(x, y, w):
  """
  Find a line with direction determined by w that has the largest margins
  """

  projected = x.dot(w)
  l = np.max( projected[ y == -1 ] )
  r = np.min( projected[ y ==  1 ] )
  b = - (l + r) / 2
  scale = 2 / (l - r)
  return w * scale, b * scale

def draw_line_with_margin(w, b, x_min, x_max):
  """
  Draw the line determined by w and b with margins
  """

  draw_line(w, b  , x_min, x_max)
  draw_line(w, b-1, x_min, x_max, color='red')
  draw_line(w, b+1, x_min, x_max, color='red')


# Exercise 2
In the code below, you can experiment with different random seed and the number of data points to observe how the solution depends on the distribution of data points. You can compare to the solutions in Exercise 1.

In [ ]:
np.random.seed(1031)

x, y = get_data(all_points=32)

# find a valid line
w = get_random_direction()
while not direction_lineary_separates_2D(x, y, w):
    w = get_random_direction()
w, b = find_line_with_margins(x, y, w)

# plot all
x_min, x_max = plot_data(x, y)
draw_line_with_margin(w, b, x_min, x_max)

Thus besides the main hyperplane (green line), we get 2 additional ones which are called margins (red lines). All the points belonging to the -1 class lay below the left margin, while all the ones that belong to 1 class lay above the right one.

Notice that 2 data points lay directly on the margins. These 2 points have a special name: support vectors, which limit the size of the margin.


*Side note:*

How can a hyperplane define margins as well?

We define hyperplane using $D+1$ preamters for $w$ and $b$. However, it can be defined using only $D$ parameters.

For instance in 2D the line:
$$w_0 x_0 + w_1 x_1 + b = 0$$
defines exactly the same line as
$$\frac{w_0}{2} x_0 + \frac{w_1}{2} x_1 + \frac{b}{2}  = 0$$
Thus, our paremteres overdeifne the hyperplane and the one free dimension defines the margins.

# Exercise 3 
Find the size of the margins of a hyperplane defined by $w$ and $b$.

Hint 1: use pen and paper

give the form of the solution
The solution should be in the form: $$\frac{const}{|w|}$$

Hint 2: x = $\alpha w , \alpha \in R$ is a vector equation of a line parallel to both margins. \
Hint 3: Find crosspoints between that line and the margins. \
Hint 4: Calculate the distance between those points. \

# Hard Margin SVM Optimisation

Thus in optimization form the problem is:
$$argmax_{w,b} \;  \frac{const}{|w|}$$ 
such that
$$w^Tx_i+ b >  1 \text{  when  } y_i =  1 $$
$$w^Tx_i+ b < -1 \text{  when  } y_i = -1 $$

This formulation can be simplified by noticing that maximizing $\frac{const}{|w|}$ is equivalent to minimizing $|w|$. Additionally, the constraints can be rewritten as $y_i(w^Tx_i+ b)>1$. Thus, the problem can be rewritten as:
$$argmin_{w,b} \;  |w|$$
such that
$$y_i(w^Tx_i+ b)>1$$

*Optimisation Sidenote:*

This is a convex minimization problem since $f(x)=|x|$ is a convex function and the constraints are linear. This means that we can find the global solution using gradient descent algorithms accounting for the inequality constraints. In practice, the solution can be found more effectively using Lagrangian dual of the problem.

\

\

\

# Softmargin SVM

What if the data is not linearly separable?

We can relax the constraints to allow for some data points to violate the margin.
Recall the constrain equaion generated by data point $i$:
$$y_i(x_iw+ b)>1$$
Thus, we should penalize the $w,b$ selection if it is violated.
$$cost_i = max(0, 1-y_i(x_iw + b))$$

You can verify that if the point lays on the correct side of the margin the cost is 0, and if it is on the other one the cost is equal to the distance to the margin.

The revised optimisation problem is:
$$argmin_{w,b} \; \text{Loss}(w,b) = 
argmin_{w,b} \;\; |w| + \lambda \frac{1}{N} \sum_{i\in\{1,..,N\}}{cost_i} $$
or
$$argmin_{w,b} \;\; |w| + \lambda \frac{1}{N} \sum_{i\in\{1,..,N\}}{max(0, 1-y_i(x_iw+ b) )} $$
where $\lambda$ is a hyperparamter. The larger it is, the more importance is given to the second term of the loss and the more the solution resembles the hard margin SVM.

Since, this is an unconstrained optimization problem we can solve it using gradient descent.
The gradient of the function is 
$$\nabla_w \text{Loss}(w,b) = \frac{w}{|w|} - \lambda \frac{1}{N} 
\sum_{i\in\{1,..,N\}}
{\mathbb{1}(1-y_i(x_iw+ b)) \: y_i x_i} $$

$$\nabla_b \text{Loss}(w,b) = - \lambda \frac{1}{N} 
\sum_{i\in\{1,..,N\}}
{\mathbb{1}( 1-y_i(x_iw+ b)) \: y_i}$$

where $\mathbb{1}$ is  a step function
$$\mathbb{1}(x)  = \begin{cases}
          1 & \text{if} & x \ge 0 \\
          0 & \text{if} & x \lt 0
      \end{cases} 
$$

# Exercise 4
Implement a function that calculates the loss given $w$, $b$, $\lambda$ and the data $x$, $y$



In [ ]:
def loss(w, b, lam, x, y):
  """
  Calculates loss of the SVM defined by w and b, given the data x, y

  :param w: a numpy array of size 2 with coefficients of the line
  :param b: scalar, the constant of the line
  :param lam: scalar hyperparameter of SVN
  :param x: numpy array of size (N, 2) with the data points
  :param y: numpy array of size (B, ) with the data labels (1 or -1)

  :return: scalar the Loss 
  """

  # write your code here

  return 0

# Excercise 5

Implement a function that calulates the gradient of Loss with respect to $w$ and $b$.

In [ ]:
def gradient(w, b, lam, x, y):
  """
  Calculates loss of the SVM defined by w and b, given the data x, y

  :param w: a numpy array of size (2,) with coefficients of the line
  :param b: scalar, the constant of the line
  :param lam: scalar hyperparameter of SVN
  :param x: numpy array of size (N, 2) with the data points
  :param y: numpy array of size (B,) with the data labels (1 or -1)

  :return: a tuple (grad_w, grad_b) 
  - grad_w: a numpy array of size (2,) with the gradient of w w.r.t. Loss
  - grad_b: a scalar of the gradient of b w.r.t. Loss
  """
  
  # write your code here
  
  return np.array([0, 0]), 0

In [ ]:
def get_alpha(w, b, grad_w, grad_b, lam, x, y):
  """
  :return: a scalar that determines the size of the step for optimisation
  """
  alpha = 1
  loss_val = loss(w, b, lam, x, y)
  for i in range(50):
    w_new = w - alpha * grad_w
    b_new = b - alpha * grad_b
    loss_new = loss(w_new, b_new, lam, x, y)
    if loss_new <= loss_val: # - alpha * 0.5 * (grad_w.dot(grad_w) + grad_b ** 2):
      return alpha    
    alpha = alpha / 1.2
  return 0

In [ ]:
lam = 200

np.random.seed(1034)
x, y = get_data(all_points=30, separable=False)
w = np.ones((2,))
b = 1

N_iter = 1000
for i in range(1, N_iter):
  loss_val = loss(w, b, lam, x, y)
  if i % (N_iter // 10) == 0:
    print("step : {:3} , Loss : {}".format(i, loss_val))

  grad_w, grad_b = gradient(w, b, lam, x, y)
  alpha = get_alpha(w, b, grad_w, grad_b, lam, x, y)
  w -= grad_w * alpha
  b -= grad_b * alpha
  
  if alpha == 0:
    print("i : {}".format(i))
    break
  
x_min, x_max = plot_data(x, y)
draw_line_with_margin(w, b, x_min, x_max)

# Excercise 6
Try different hyperparameters lambda with separable and non-separable data. Try lambda ranging from 2 to 200.

Write your observations (around 30 words).